# SentimentOps — Exploratory Data Analysis

This notebook explores the cleaned Amazon product reviews dataset to understand class distributions, review characteristics, and vocabulary patterns across sentiment classes.

**Dataset**: `data/processed/cleaned_reviews.csv` (output of `src/preprocess.py`)

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLEANED_DATASET_PATH, NOTEBOOKS_FIGURES_DIR

NOTEBOOKS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Sentiment Class Distribution

In [ ]:
order = ['positive', 'neutral', 'negative']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
counts = df['sentiment'].value_counts().reindex(order)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(order, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for i, val in enumerate(counts.values):
    ax.text(i, val + 5, str(val), ha='center', fontweight='bold', fontsize=12)
ax.set_title('Sentiment Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(NOTEBOOKS_FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 2. Review Length Distribution

In [ ]:
df['review_length'] = df['review_text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 5))
for sentiment, color in zip(order, colors):
    subset = df[df['sentiment'] == sentiment]['review_length']
    ax.hist(subset, bins=50, alpha=0.6, label=sentiment, color=color)
ax.set_title('Review Length Distribution by Sentiment', fontweight='bold')
ax.set_xlabel('Number of Words')
ax.set_ylabel('Frequency')
ax.legend()
ax.set_xlim(0, df['review_length'].quantile(0.98))
fig.tight_layout()
fig.savefig(NOTEBOOKS_FIGURES_DIR / 'review_length_distribution.png', dpi=150)
plt.show()

## 3. Top 20 Most Common Words per Sentiment

In [ ]:
stopwords = set(
    'i me my myself we our ours ourselves you your yours yourself yourselves '
    'he him his himself she her hers herself it its itself they them their theirs '
    'themselves what which who whom this that these those am is are was were be '
    'been being have has had having do does did doing a an the and but if or '
    'because as until while of at by for with about against between through during '
    'before after above below to from up down in out on off over under again further '
    'then once here there when where why how all both each few more most other some '
    'such no nor not only own same so than too very s t can will just don should now '
    'read br also would could one two get got like really much even still use go going '
    'make know think see come take want'.split()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, sentiment, color in zip(axes, order, colors):
    texts = df[df['sentiment'] == sentiment]['review_text'].str.cat(sep=' ')
    words = [w for w in texts.split() if w not in stopwords and len(w) > 2]
    wc = Counter(words).most_common(20)
    if wc:
        w, c = zip(*wc)
        ax.barh(range(len(w)), c, color=color)
        ax.set_yticks(range(len(w)))
        ax.set_yticklabels(w)
        ax.invert_yaxis()
    ax.set_title(f'Top 20 — {sentiment.title()}', fontweight='bold')
fig.tight_layout()
fig.savefig(NOTEBOOKS_FIGURES_DIR / 'top_words_per_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, sentiment in zip(axes, order):
    texts = df[df['sentiment'] == sentiment]['review_text'].str.cat(sep=' ')
    wc = WordCloud(width=800, height=400, background_color='white', max_words=100,
                   stopwords=stopwords, colormap='viridis').generate(texts)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment.title()}', fontweight='bold')
    ax.axis('off')
fig.suptitle('Word Clouds by Sentiment', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(NOTEBOOKS_FIGURES_DIR / 'wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Average Rating by Product Category

In [ ]:
if 'product_category' in df.columns:
    df['primary_category'] = df['product_category'].fillna('Unknown').apply(
        lambda x: x.split(',')[0].strip()
    )
    cat_ratings = df.groupby('primary_category')['rating'].agg(['mean', 'count'])
    cat_ratings = cat_ratings[cat_ratings['count'] >= 5].sort_values('mean', ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, len(cat_ratings) * 0.4)))
    ax.barh(cat_ratings.index, cat_ratings['mean'],
            color=sns.color_palette('coolwarm', len(cat_ratings)))
    ax.set_xlabel('Average Rating')
    ax.set_title('Average Rating by Product Category', fontweight='bold')
    ax.set_xlim(0, 5.5)
    fig.tight_layout()
    fig.savefig(NOTEBOOKS_FIGURES_DIR / 'avg_rating_by_category.png', dpi=150)
    plt.show()
else:
    print('No product_category column found')